### Developing human heart - ligand-receptor communication in regions, subregions, ages and between cell types (mp_heart_dvlp04)

**Paper**: Lázár E, Mauron R, Andrusivová Ž, Foyer J, He M, Larsson L, Shakari N, Salas SM, Avenel C, Sariyar S, Hansen JN, Vicari M, Czarnewski P, Braun E, Li X, Bergmann O, Sylvén C, Lundberg E, Linnarsson S, Nilsson M, Sundström E, Adameyko I, Lundeberg J. *Spatiotemporal gene expression and cellular dynamics of the developing human heart.* Nat Genet. 2025 Nov;57(11):2756-2771. doi: [10.1038/s41588-025-02352-6](https://doi.org/10.1038/s41588-025-02352-6). PMID: 41162788; PMCID: PMC12597827.

Code: [github.com/rmauron/HDCA_heart_dev](https://github.com/rmauron/HDCA_heart_dev) (Zenodo 15912657) - Data: Mendeley [part 1](https://doi.org/10.17632/fhtb99mdzd.1) (Space Ranger, Cell Ranger, metadata) and [part 2](https://doi.org/10.17632/w65jtfsvpr.1) (Seurat RDS) - [Interactive viewer](https://hdcaheart.serve.scilifelab.se/web/index.html)

LIANA+ bivariate (spatial co-expression of ligand and receptor), cell-type-resolved sender -> receiver scores and the reference rank_aggregate, as in **mp_lymphoid04** / **mp_mouse_brain04**, **per region** (mp_heart_dvlp01), **per subregion** (mp_heart_dvlp03) and, new here, **per age group** (w6-w7, w8-w9, w10-w12).

**Question**: which ligand-receptor (LR) pairs are spatially active **in each region / subregion / developmental stage** of the heart, and **between which cell types** do they act? The paper predicted niche-level interactions with LIANA on the scRNA data (e.g. in the developing valves, epicardium and conduction system); here the spatial data are used directly.

#### Libs: **src/libs**

- **single_cell_lib**
  - Class **SingleCell**: generic Scanpy, Squidpy, cell2location and LIANA methods
  - **DEFAULT_PARAMS**
- **single_cell_heart_dvlp_lib** (new)
  - Class **SingleCellHeart(SingleCell)**: developing-heart methods (sections and ages, the paper's stereoscope proportions and spatial clusters, gene x age panels, regions vs the paper, temporal trends)
  - **HEART_PARAMS**, **HEART_LINEAGE** (31 cell types in 9 lineages), **HEART_GENES** (20 heart-development genes), **HEART_FILES** (one file registry per notebook: 'heart01' ... 'heart04')

| section | content |
|---|---|
| 1 | spots, regions, subregions, ages, abundances, signatures |
| 2 | spatial weights (Gaussian kernel per section, bandwidth in spot spacings) |
| 3 | LIANA bivariate: global Moran's R and local score of every LR pair |
| 4 | LR pairs enriched in each region and subregion |
| 5 | LR pairs enriched in each age group |
| 6 | between cell types: sender -> receiver (whole tissue) |
| 7 | between cell types inside each region / subregion / age group |
| 8 | reference LIANA (rank_aggregate) + cell-type co-localisation |
| 9 | consensus per region / subregion |
| 10 | tables |

**Data**: the 38 sections (69,115 spots), regions of **heart01_region_labels.tsv**, subregions of mp_heart_dvlp03 (**heart03_subregions/**), the scRNA reference (31 types). LR resource: LIANA **consensus** (human). Figures and tables have the prefix **heart04_**.

### Critique of the request and how it is handled

- **What a Visium LR score can mean.** A spot (55 um, 100 um spacing) holds ~10-30 cells of several types. The bivariate local score says that ligand and receptor are co-expressed **in a neighbourhood** (~2 rings, ~200 um), not that two cells signal.
- **38 sections.** The spatial kernel is built **per section** (block-diagonal: spots of different sections are never neighbours); global scores pool all sections.
- **Ages.** An LR pair ranked high in w10-w12 is higher in those sections than in the others; with 2-3 hearts per stage and different section planes (later hearts only partially captured), age contrasts mix development with anatomy. Read them together with the region contrasts.
- **Statistics.** Neighbouring spots are not independent: p-values are optimistic; groups and links are ranked by effect sizes (AUROC, score, z) with parameter cutoffs.
- **Between cell types.** Visium does not see cell types; the counts of each spot are split across cell types by abundance x reference signature (**calc_celltype_expr**). Signatures: cell2location reference (**means**) when the model exists, otherwise mean counts per cell type of the reference. Attribution follows the reference.
- **Uninformative pairs** (MHC, B2M, intracellular 'ligands'; **lr_display_exclude**) are computed but not listed. **R18** is excluded (**subregion_skip**).

## Loading packages

In [ ]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import gc

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42          # editable text in PDFs
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'dpi': 90}   # inline display only; files are saved at 300 dpi
import plotly.io as pio
pio.renderers.default = "notebook_connected"

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_heart_dvlp_lib import SingleCellHeart, HEART_PARAMS, HEART_LINEAGE, HEART_GENES, HEART_FILES
from libs.single_cell_lib import cmap_byr, cmap_lbwr
from libs.matplotlib_figure_style import apply_figure_style
from libs.Basic import create_dir

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print({p: version(p) for p in ['liana', 'scanpy', 'anndata', 'plotly']})

## Parameters

Class defaults (**DEFAULT_PARAMS** + **HEART_PARAMS**: LIANA values shared with mp_lymphoid04 / mp_mouse_brain04); files **HEART_FILES['inputs']** + **['heart04']**. Caching: every table method reads its file when it exists and **force=False**.

In [ ]:
force=False
verbose=False

apply_figure_style()
sch = SingleCellHeart(ROOT0, analysis='heart04')
ref_groupby, ref_symbol_col = sch.params['celltype_col'], sch.params['ref_symbol_col']
print(sch)
print({k: sch.params[k] for k in ['liana_resource', 'liana_bandwidth_spots', 'liana_nz_prop', 'lr_group_min_auroc', 'lr_display_exclude']})
sch.calc_df_files()

### 1. Spots, regions, subregions, ages and cell-type abundances

**prepare_expression**: symbols as var_names, raw counts in **layers['counts']**, X = log1p(CP10k). Subregions: **run_subregions(force=False)** reads the cached subsets of mp_heart_dvlp03.

In [ ]:
df_regions = sch.read_tsv('region_labels', verbose=verbose)
source = df_regions['abundance_source'].iloc[0]
adata_map = sch.read_h5ad_slots('c2l_map', skip=('layers',)) if source == 'c2l' else sch.read_h5ad('visium', set_self=False)
adata_map = sch.remove_mt_genes(adata_map)
names_auto = sch.add_region_labels_from_table(df_regions, adata_map)
df_region_info = sch.read_region_info(source=source)
region_name = dict(zip(df_region_info.region_id, df_region_info.region_name)) if df_region_info is not None else names_auto
df_abund = sch.get_abundance(adata_map, source=source)

adata_ref = sch.filter_reference(sch.read_h5ad('reference', set_self=False))
df_sig = sch.reference_signatures_symbols(adata_ref, summary='means')       # genes (symbols) x cell types
print("signatures:", df_sig.attrs['source'], df_sig.shape)

adata_expr = sch.prepare_expression(adata_map)
subs, _ = sch.run_subregions(adata_expr, force=False, verbose=verbose)       # obs['sub_region_id'] from mp_heart_dvlp03
celltypes = [c for c in df_abund.columns if c in df_sig.columns]

df_sub03 = sch.read_tsv('subregion_table_heart03', verbose=verbose)
sub_supported = df_sub03.loc[df_sub03.status.str.startswith('supported'), 'sub_region_id'].tolist()
skip = set(sch.params['subregion_skip'])
groups = {'region': sch._groups(adata_expr, 'region_label', skip=skip),
          'subregion': sch._groups(adata_expr, 'sub_region_id'),
          'age_group': sch._groups(adata_expr, 'age_group')}
sections_rep = sch.representative_sections()
print(adata_expr.shape, '|', len(celltypes), 'cell types |', groups['region'].nunique(), 'regions |',
      groups['subregion'].nunique(), 'subregions (', len(sub_supported), 'supported in 03 ) |', groups['age_group'].nunique(), 'age groups')

### 2. Spatial weights

Gaussian kernel **w = exp(-d^2 / 2bw^2)** with **bw = liana_bandwidth_spots x spot spacing**, built **per section**, self weight 1, weights < **liana_cutoff** removed.

In [ ]:
force=False
verbose=False
df_w = sch.spatial_weights(adata_expr, force=force, verbose=verbose)
W = adata_expr.obsp['spatial_connectivities']
df_w.head(10)

### 3. LIANA bivariate: spatially co-expressed LR pairs

**li.mt.bivariate** (resource **liana_resource**): global **Moran's R** (permutation p, BH FDR) and the **local cosine** score of every pair at every spot, with categories (1 = high-high) and local p-values. Ligand and receptor (all complex subunits) in >= **liana_nz_prop** of the spots. Cached in **root_result**.

In [ ]:
force=False
verbose=False
df_inter, adata_lr = sch.liana_bivariate(adata_expr, force=force, verbose=verbose)
df_lr_sig = sch.lr_significant(df_inter)
print(f"LR pairs tested: {len(df_inter)}; Moran's R > 0 at FDR <= {sch.params['liana_fdr']}: {len(df_lr_sig)}")
df_lr_sig[['ligand', 'receptor', 'ligand_props', 'receptor_props', 'morans', 'morans_fdr']].head(20).round(3)

In [ ]:
force=False
verbose=False
lr_show = sch._lr_display(df_lr_sig.index)[:8]
for lib in [sch.params['library_main'], sections_rep[0], sections_rep[-1]]:
    fig = sch.plot_lr_spatial(adata_lr, adata_expr, lr_show, library_id=lib, ncols=4, figsize=(18, 8), fontsize_title=14,
                              fontsize_suptitle=16, fontsize_colorbar=12, fname='fig_lr_top', force=force, verbose=verbose)
    plt.show()

### 4. LR pairs enriched in each region and subregion

**calc_df_lr_by_group**: per group, the local score of every significant pair in the group vs the rest of the tissue: **AUROC**, mean local score in / out, fraction of high-high spots in / out; **enriched** = AUROC >= **lr_group_min_auroc** and more high-high spots than elsewhere.

In [ ]:
force=False
verbose=False
df_lr_region = sch.calc_df_lr_by_group(adata_lr, groups['region'], df_lr_sig.index, fname='lr_by_region', force=force, verbose=verbose)
df_lr_sub = sch.calc_df_lr_by_group(adata_lr, groups['subregion'], df_lr_sig.index, fname='lr_by_subregion', force=force, verbose=verbose)

df_n = pd.DataFrame({'n_enriched': df_lr_region[df_lr_region.enriched].groupby('group').size()}) \
         .reindex(sorted(groups['region'].dropna().unique())).fillna(0).astype(int)
df_n['region_name'] = df_n.index.map(region_name)
df_n['top_lr'] = sch.calc_df_lr_group_top(df_lr_region)
df_n

In [ ]:
force=False
verbose=False
fig = sch.plotly_lr_group_heatmap(df_lr_region, n_per_group=3, group_names=region_name, height=950, width=1800, fontsize_title=22,
                                  fontsize_xticks=13, fontsize_yticks=13, fontsize_colorbar=14, fontsize_hover=14,
                                  fname='fig_lr_region_hm', force=force, verbose=verbose)
fig.show()

The top enriched pair of each region on the w8 section (region outlined):

In [ ]:
force=False
verbose=False
top1 = df_lr_region[df_lr_region.enriched & df_lr_region.interaction.isin(set(sch._lr_display(df_lr_region.interaction.unique())))] \
         .groupby('group').head(1)
reg_lab = adata_expr.obs['region_label'].astype(str).values
ncols = 5
nrows = -(-len(top1) // ncols)
lib = sch.params['library_main']
fig = sch.plot_lr_spatial(adata_lr, adata_expr, list(top1.interaction), library_id=lib, ncols=ncols, figsize=(4.2 * ncols, 4 * nrows),
                          titles=[f"{g}: {i} (AUROC {a:.2f})" for g, i, a in zip(top1.group, top1.interaction, top1.auroc)],
                          outlines=[reg_lab == g for g in top1.group], fontsize_title=12, fontsize_suptitle=16, fontsize_colorbar=12,
                          fname='fig_lr_regions', force=force, verbose=verbose)
plt.show()

Subregions: the **supported** subregions of mp_heart_dvlp03.

In [ ]:
force=False
verbose=False
df_n_sub = df_lr_sub[df_lr_sub.group.isin(sub_supported)]
print(f"{df_n_sub[df_n_sub.enriched].group.nunique()} of {len(sub_supported)} supported subregions have >= 1 enriched LR pair")
fig = sch.plotly_lr_group_heatmap(df_lr_sub, n_per_group=2, groups=[g for g in sub_supported if g in set(df_lr_sub.group)],
                                  height=1400, width=1900, fontsize_title=22, fontsize_xticks=13, fontsize_yticks=13,
                                  fontsize_colorbar=14, fontsize_hover=14, fname='fig_lr_sub_hm', force=force, verbose=verbose)
fig.show()

### 5. LR pairs by age group

The same test with the three developmental stages as groups (w6-w7: 11 sections, w8-w9: 12, w10-w12: 15). Stage contrasts are much weaker than region contrasts: on this data no pair reaches **lr_group_min_auroc** (0.65; maximum about 0.64), because every stage contains all tissue compartments. The pairs are therefore **ranked** by AUROC per stage (descriptive, no cutoff), with the region of each pair's strongest regional enrichment: an age-ranked pair often reflects a compartment that is larger at that stage (e.g. cushion and outflow-tract matrix early, coronary and fibroblast tissue late) as much as a change of signalling within one compartment.

In [ ]:
force=False
verbose=False
df_lr_age = sch.calc_df_lr_by_group(adata_lr, groups['age_group'], df_lr_sig.index, fname='lr_by_age', force=force, verbose=verbose)
print(df_lr_age.groupby('group').auroc.describe().round(3))
best_region = df_lr_region.sort_values('auroc', ascending=False).drop_duplicates('interaction').set_index('interaction')['group']
keep = set(sch._lr_display(df_lr_age.interaction.unique()))
df_age_top = df_lr_age[df_lr_age.interaction.isin(keep)].sort_values('auroc', ascending=False).groupby('group').head(10).copy()
df_age_top['best_region'] = df_age_top.interaction.map(best_region)
df_age_top['best_region_name'] = df_age_top.best_region.map(region_name)
df_age_top[['group', 'interaction', 'auroc', 'mean_in', 'mean_out', 'hh_frac_in', 'hh_frac_out', 'best_region', 'best_region_name']].round(3)

In [ ]:
# AUROC of the top-ranked pairs of every stage in all three stages (0.5 = as in the other stages)
fig = sch.plot_lr_auroc_by_age_group(df_lr_age, df_age_top, fname='heart04_lr_by_age_group_top.png', force=force, verbose=verbose)
plt.show()

### 6. Between cell types: sender -> receiver (whole tissue)

**calc_celltype_expr**: counts of every spot split across cell types (abundance x signature), log1p(CP10k). **calc_df_celltype_lr** for the **lr_top_n** significant pairs with the highest Moran's R: **score[A, B] = mean over spots of (W_norm · L_A) x R_B**; null: receptor spots permuted (**c2l_lr_n_perms**) -> z; **score_share** = share of the pair's total over all A, B.

In [ ]:
force=False
verbose=False
df_lr_top = df_lr_sig.head(sch.params['lr_top_n'])[['ligand', 'receptor']]
genes = sorted({s for e in list(df_lr_top.ligand) + list(df_lr_top.receptor) for s in e.split('_')})
E, genes_E, celltypes_E = sch.calc_celltype_expr(adata_expr, df_abund.loc[adata_expr.obs_names, celltypes], df_sig[celltypes], genes)
print('E (spots x cell types x genes):', E.shape, '| LR genes without signature:', sorted(set(genes) - set(genes_E)))
df_ct, df_group_ct = sch.calc_df_celltype_lr(E, genes_E, celltypes_E, df_lr_top, W, groups=groups, force=force, verbose=verbose)
df_best = df_ct[df_ct.rank_in_lr == 1].set_index('interaction')
df_best = df_best.loc[[i for i in df_lr_top.index if i in df_best.index]]
print(df_ct.interaction.nunique(), 'LR pairs x', len(celltypes_E), 'x', len(celltypes_E), 'sender-receiver')
df_best[['sender', 'receiver', 'score', 'score_share', 'z']].head(20).round(3)

In [ ]:
force=False
verbose=False
lr_panels = sch._lr_display(df_best.index)[:6]
fig = sch.plot_sender_receiver_heatmaps(df_ct, lr_panels, n_top=10, ncols=3, panel_size=(8, 7.0), fontsize_title=16, fontsize_ticks=12,
                                        fontsize_label=12, fontsize_colorbar=12, fname='fig_sender_recv', force=force, verbose=verbose)
plt.show()

### 7. Between cell types inside each region / subregion / age group

The same score over the spots of each group; receptor spots permuted **within** the group (**c2l_lr_group_n_perms**) -> z; **ratio** = group score / whole-tissue score. **calc_df_group_celltype_links**: per group and sender -> receiver, the number of LR pairs with z >= **c2l_lr_min_z** and ratio >= 1.

In [ ]:
force=False
verbose=False
df_links = sch.calc_df_group_celltype_links(df_group_ct, force=force, verbose=verbose)
lev = df_group_ct.drop_duplicates('group').set_index('group').level
df_links_reg = df_links[df_links.group.map(lev) == 'region']
df_links_reg[~df_links_reg.self_link].groupby('group').head(2)[['group', 'link', 'n_lr', 'lr']]

In [ ]:
force=False
verbose=False
fig = sch.plotly_group_celltype_links(df_links_reg, n_links=30, group_names=region_name, groups=sorted(df_links_reg.group.unique()),
                                      height=1000, width=1600, fontsize_title=22, fontsize_xticks=14, fontsize_yticks=14,
                                      fontsize_colorbar=14, fontsize_hover=14, fname='fig_links_region', force=force, verbose=verbose)
fig.show()

In [ ]:
force=False
verbose=False
df_links_sub = df_links[df_links.group.isin(sub_supported)]
groups_sub = [g for g in sub_supported if g in set(df_links_sub.group)]
fig = sch.plotly_group_celltype_links(df_links_sub, n_links=30, groups=groups_sub, width=1600, height=1300, fontsize_title=22,
                                      fontsize_xticks=13, fontsize_yticks=13, fontsize_colorbar=14, fontsize_hover=14,
                                      fname='fig_links_sub', force=force, verbose=verbose)
fig.show()

In [ ]:
# age groups: top sender -> receiver links per stage
df_links_age = df_links[df_links.group.map(lev) == 'age_group']
df_links_age[~df_links_age.self_link].groupby('group').head(6)[['group', 'link', 'n_lr', 'lr']]

### 8. Reference LIANA (rank_aggregate) + cell-type co-localisation

**liana_rank_aggregate_reference**: rank_aggregate (CellPhoneDB, Connectome, log2FC, NATMI, SCA, CellChat-like) on the reference (up to **ref_lr_cells_per_type** cells per type; symbols from **var['symbol']**). Non-spatial, so each source -> target is annotated with **co-localisation** (**calc_df_celltype_coloc**). Specific and strong = specificity_rank <= **ref_lr_spec_rank** and magnitude_rank <= **ref_lr_mag_rank**; **spatially supported** = co-localisation r >= **coloc_min** or group share >= **coloc_group_min**.

In [ ]:
force=False
verbose=False
df_liref = sch.liana_rank_aggregate_reference(adata_ref, ref_groupby, symbol_col=ref_symbol_col, force=force, verbose=verbose)
df_coloc, coloc_region = sch.calc_df_celltype_coloc(df_abund.loc[adata_expr.obs_names, celltypes_E], W, groups['region'])
df_ref_sup = sch.calc_df_ref_lr_support(df_liref, df_coloc, coloc_region, force=force, verbose=verbose)
print(f"reference interactions: {len(df_liref)}; specific & strong: {len(df_ref_sup)}; spatially supported: {int(df_ref_sup.spatially_supported.sum())}")
df_ref_sup[df_ref_sup.spatially_supported & df_ref_sup.listed][['source', 'target', 'interaction', 'magnitude_rank', 'specificity_rank',
                                                                'coloc', 'best_group', 'coloc_group']].head(20).round(3)

In [ ]:
force=False
verbose=False
fig = sch.plotly_coloc_heatmap(df_coloc, height=1100, width=1500, fontsize_title=22, fontsize_ticks=14, fontsize_colorbar=14,
                               fontsize_hover=14, fname='fig_coloc', force=force, verbose=verbose)
fig.show()

### 9. Consensus per region / subregion

**calc_df_group_ccc**: for each group, its top enriched LR pairs (section 4) with the best sender -> receiver of the pair **in the group** (section 7: z, ratio vs tissue), and the reference: **ref_same_link** = rank_aggregate finds the same LR between that sender and receiver, **ref_links** = best spatially supported reference source -> target of the pair.

In [ ]:
force=False
verbose=False
df_ccc_region = sch.calc_df_group_ccc(df_lr_region, df_group_ct[df_group_ct.level == 'region'], df_ref_sup,
                                      group_names=region_name, fname='ccc_by_region', force=force, verbose=verbose)
print('LR pairs with a cell-type link in the group:', int((df_ccc_region.c2l_sender.fillna('') != '').sum()), 'of', len(df_ccc_region),
      '| same link in the reference:', int(df_ccc_region.ref_same_link.sum()))
df_ccc_region.round(2)

In [ ]:
force=False
verbose=False
df_ccc_sub = sch.calc_df_group_ccc(df_lr_sub[df_lr_sub.group.isin(sub_supported)], df_group_ct[df_group_ct.level == 'subregion'],
                                   df_ref_sup, n=3, fname='ccc_by_subregion', force=force, verbose=verbose)
df_ccc_sub.round(2)

### 10. Tables

All in **root_result** with prefix **heart04_**: spatial weights, bivariate global table, LR by region / subregion / age group, cell-type LR (tissue, by group), cell-type links by group, reference support, consensus by region / subregion; caches: bivariate h5ad, reference rank_aggregate.

### Notes for interpretation

- **AUROC** compares a group with the rest of the tissue; a pair can be enriched in a group because its producers are concentrated there (composition), which is the point of the cell-type sections.
- **Sender -> receiver** calls follow the reference signatures x abundances; they say which types *can* carry ligand and receptor *where*, not that they signal.
- **z** values ignore spatial autocorrelation and are large for any co-localised pair; compare links by score, ratio and n_lr.
- **Abundance source**: with the stereoscope proportions (no cell2location map yet) the abundances are proportions, so the split of the counts is the same as with cell numbers only when the spots have similar cell density.